# Variant Effect Scanner: other PLMs on Colab (Task 2.2)
Scans all 126 proteins with **ESM-2 650M** and **ESM-C 600M**, for entropy baselines. Runtime > Change runtime type > **T4 GPU**.
Resumable: results go to Drive, and re-running skips finished proteins. Push your latest local commits to the `saksham` branch first.

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_URL = "https://github.com/A-k-s-h-a-t-G-u-p-t-a/Variant-effect-scanner.git"
BRANCH = "saksham"
REPO = "/content/drive/MyDrive/Variant-effect-scanner"
if os.path.isdir(REPO + "/.git"):
    !git -C "{REPO}" fetch origin
    !git -C "{REPO}" checkout {BRANCH}
    !git -C "{REPO}" pull --ff-only origin {BRANCH}
else:
    !git clone -b {BRANCH} "{REPO_URL}" "{REPO}"
%cd {REPO}
!git log --oneline | head -3
!ls scripts

In [ ]:
!pip install -q transformers==5.19.0 scikit-learn scipy pandas tqdm

## ESM-2 650M (`facebook/esm2_t33_650M_UR50D`)
Prints the token-offset check for this model first; it must say verified.

In [ ]:
!python scripts/scan_model.py --model facebook/esm2_t33_650M_UR50D --tag esm2_650m --device cuda

## ESM-C 600M (`biohub/ESMC-600M-hf`)

In [ ]:
!python scripts/scan_model.py --model biohub/ESMC-600M-hf --tag esmc_600m --device cuda

## Check, rebuild table, download

In [ ]:
import glob
for tag in ["esm2_650m", "esmc_600m"]:
    n = len(glob.glob(f"results/logprobs_{tag}/*.npy"))
    print(tag, n, "of 126 scanned")
!python scripts/build_site_table.py

In [ ]:
!zip -qr models_scans.zip results/logprobs_esm2_650m results/logprobs_esmc_600m
from google.colab import files
files.download("models_scans.zip")